# Background

Author: Paul Inkenbrandt
Date: 11/1/2026

Purpose: Batch-run every footprint model in `fluxfootprints` (`ffp_xr`, `ffp`, `km`, `ls`,
`wang`) for multiple stations and save every result as a Zarr store.

For each station, each month in the analysis period, and each model, the notebook writes:

| Store | Written by | Models | Contents |
|---|---|---|---|
| `{station}/{model}/{station}_{model}_{yyyymm}_results.zarr` | `model.to_zarr()` | all | the model's full results Dataset (climatology, plus the per-timestep footprint for `ffp`/`ffp_xr`) |
| `{station}/{model}/{station}_{model}_{yyyymm}_summaries.zarr` | `export_summaries_zarr()` | `ffp`, `ffp_xr` | daily/monthly mean and ET-weighted footprints, georeferenced to local UTM |
| `{station}/{station}_monthly_climatology.zarr` | this notebook | all | one variable per model, dims `(time_monthly, y, x)`, every model regridded onto the same grid |

A `run_manifest.csv` in `OUT_DIR` lists every station/month/model run, its status, runtime,
and the stores it produced.

Input data is one combined parquet file per station, in the schema of
`input_data/US-UTL_combined_data.parquet`:

- a `DatetimeIndex` named `datetime_start` (half-hourly, local time);
- flux/met columns with instrument suffixes, e.g. `ustar_1_1_1`, `mo_length_1_1_1`, `ws_1_1_1`,
  `v_sigma_1_1_1`, `wd_1_1_1`, `et_1_1_1`, `pblh_f`;
- footprint inputs already attached: `measurement_height`, `crop_height_m`, `zm`, `z0`;
- `nldas_et`, NLDAS reference ET (mm/hr) used for ET weighting.

Station coordinates come from `stations_latest_locations.geojson` (features keyed by `stationid`).

Things to know before running:

- **Only `ffp` and `ffp_xr` have a per-timestep footprint**, so only they get daily/monthly
  `summaries` stores. `km`, `ls` and `wang` only produce a climatology.
- **`ls` is Monte-Carlo** (cost ~ `n_particles x n_timesteps`, roughly a second per
  timestep at 5000 particles). Each month's `ls` run is therefore limited to `LS_MAX_TIMESTEPS`
  evenly spaced rows with `LS_N_PARTICLES` particles. Set `LS_MAX_TIMESTEPS = None` to use every
  row, or drop `"ls"` from `MODEL_TYPES`.
- **`wang` only uses unstable rows** (`ol < 0`); months with too few are recorded as failed in
  the manifest rather than stopping the run.
- Models run month by month and are released after each month so memory stays bounded.

# Parameters and Libraries

In [ ]:
import gc
import json
import time
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import CRS, Transformer

sys.path.append("../../src")
from fluxfootprints import write_zarr
from fluxfootprints.ffp_daily_monthly_helper import (
    build_climatology,
    choose_utm_epsg,
    compute_aerodynamic_params,
    export_summaries_zarr,
    summarize_periods,
)

In [ ]:
# Stations to run. None = every station in STATIONS_GEOJSON that has an input file.
STATION_IDS = None   # e.g. ["US-UTL", "US-UTW"]

# Station lat/lon; features keyed by a "stationid" (or legacy "Site ID") property
STATIONS_GEOJSON = Path("../../stations_latest_locations.geojson")
# One combined parquet per station, in the schema of input_data/US-UTL_combined_data.parquet
DAT_PATH_TEMPLATE = "input_data/{sid}_combined_data.parquet"
MISSING_VALUE = -9999   # logger fill value, replaced with NaN on load

# ET weights [mm/hr]: the first of these columns with data in the analysis period is used.
# nldas_et = NLDAS reference ET carried in the combined file, et_1_1_1 = tower ET.
ET_WEIGHT_COLUMNS = ["nldas_et", "et_1_1_1"]

# zm/z0 are taken from the file's zm/z0 columns. Rows where they are missing are filled
# from measurement_height and crop_height_m with this compute_aerodynamic_params preset.
VEG_TYPE = "alfalfa"

OUT_DIR = Path("output/zarr_runs")

# Analysis period (include the time of day) and daytime hours, 8:00-20:00 follows Volk (2023).
# Set START/END to None to use each station's full record.
START = "2025-06-01 00:00"
END   = "2025-07-31 23:30"
START_TIME = 8
END_TIME = 20

DX, DY = 10.0, 10.0   # output raster resolution [m]
DOMAIN = (-600.0, 600.0, -600.0, 600.0)
BL_HEIGHT = 2000.0    # boundary-layer height [m] for ffp/ffp_xr; or "pblh_f" to use the file's column
# Allow ffp/ffp_xr rows inside the roughness sublayer (crop height close to measurement height).
# False requires zm > 27.5 z0; True only requires zm > 20 z0 (Kljun et al. 2015, Eq. 27).
# ffp also applies RSL corrections to the crosswind spread; ffp_xr only drops the check.
FFP_RSLAYER = True

MODEL_TYPES = ["ffp_xr", "ffp", "km", "ls", "wang"]
SUMMARY_MODELS = {"ffp_xr", "ffp"}   # models with a per-timestep footprint
LS_N_PARTICLES = 5000                # ls_footprint_adapter default is 20000
LS_MAX_TIMESTEPS = 100               # ls runs one particle simulation per row; evenly subsample to this many (None = all)
MIN_VALID_TIMESTEPS = 10             # skip a month/model with fewer usable rows

WRITE_FULL_RESULTS = True   # per-month model.to_zarr(); includes the (time, x, y) stack for ffp/ffp_xr
WRITE_SUMMARIES = True      # per-month export_summaries_zarr() for SUMMARY_MODELS
WRITE_COMBINED = True       # per-station monthly climatology store, all models on one grid

# Column mapping shared by every model
COLUMNS = dict(
    ustar="ustar_1_1_1",
    ol="mo_length_1_1_1",
    umean="ws_1_1_1",
    sigmav="v_sigma_1_1_1",
    wind_dir="wd_1_1_1",
    zm="zm",
    z0="z0",
)
REQ_FIELDS = list(COLUMNS.values()) + ([BL_HEIGHT] if isinstance(BL_HEIGHT, str) else [])

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Station coordinates from the station GeoJSON
with open(STATIONS_GEOJSON) as f:
    features = json.load(f)["features"]
station_coords = {
    feat["properties"].get("stationid", feat["properties"].get("Site ID")): feat["geometry"]["coordinates"]   # [lon, lat]
    for feat in features
}

station_ids = STATION_IDS if STATION_IDS is not None else sorted(station_coords)
stations = []
for sid in station_ids:
    dat_path = Path(DAT_PATH_TEMPLATE.format(sid=sid))
    if sid not in station_coords:
        print(f"{sid}: no coordinates in {STATIONS_GEOJSON.name}, skipping")
        continue
    if not dat_path.exists():
        if STATION_IDS is not None:
            print(f"{sid}: input file {dat_path} not found, skipping")
        continue
    lon, lat = station_coords[sid]
    stations.append({"station": sid, "lat": lat, "lon": lon, "dat_path": dat_path})

stations_df = pd.DataFrame(stations)
assert len(stations_df) > 0, "No stations with input data found - check STATION_IDS and DAT_PATH_TEMPLATE"
stations_df

## Helper Functions

- `load_station` reads one station's combined parquet file, replaces `MISSING_VALUE` with NaN,
  drops duplicate timestamps, keeps the file's `zm`/`z0` (filling gaps from `measurement_height`
  and `crop_height_m`), subsets to the analysis period and daytime hours, and attaches an
  `et_weight` column from the first of `ET_WEIGHT_COLUMNS` with data.
- `ffp_valid_rows` counts rows that pass the FFP input filter and Kljun et al. (2015) validity
  bounds (u\* > 0.1 m/s, zm/L >= -15.5, zm < 0.8 h, and zm > 27.5 z0, or zm > 20 z0 with
  `FFP_RSLAYER = True`). When none pass, `ffp` and
  `ffp_xr` return an all-zero climatology, so the count is printed per month and added to the
  failure message.
- `regrid` puts a model climatology onto the common `DOMAIN`/`DX`/`DY` grid. `ffp`, `ffp_xr`
  and `km` already match it; `ls` (histogram bin centres) and `wang` (own x axis) don't. Cells
  outside a model's native grid are filled with 0 and the result is renormalized to integrate
  to 1 (same approach as
  [footprint_model_comparison_all_sites.ipynb](footprint_model_comparison_all_sites.ipynb)).
- `write_combined_store` writes the per-station store holding every model's monthly
  climatology, with UTM coordinates and a CF `spatial_ref` like `export_summaries_zarr`.

In [ ]:
common_x = np.arange(DOMAIN[0], DOMAIN[1] + DX, DX)
common_y = np.arange(DOMAIN[2], DOMAIN[3] + DY, DY)
cell_area = DX * DY


def load_station(row) -> tuple[pd.DataFrame, str | None]:
    """Daytime rows in the analysis period, and the column used for ET weights."""
    df = pd.read_parquet(row.dat_path)
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.set_index("datetime_start")
    df.index = pd.to_datetime(df.index)
    df = df[df.index.notna()].sort_index()
    df = df[~df.index.duplicated(keep="first")]
    df = df.replace(MISSING_VALUE, np.nan)

    # Keep the file's zm/z0; fill gaps from measurement and crop height
    if {"measurement_height", "crop_height_m"} <= set(df.columns):
        aero = compute_aerodynamic_params(
            df,
            inst_height="measurement_height",
            crop_height="crop_height_m",
            veg_type=VEG_TYPE,
        )
        for col in ("zm", "z0"):
            df[col] = df[col].fillna(aero[col]) if col in df.columns else aero[col]

    df = df.loc[START:END]
    df = df[(df.index.hour >= START_TIME) & (df.index.hour <= END_TIME)].copy()

    et_col = next((c for c in ET_WEIGHT_COLUMNS if c in df.columns and df[c].notna().any()), None)
    # NLDAS reference ET dips slightly below 0 around dawn/dusk; negative weights are meaningless
    df["et_weight"] = df[et_col].clip(lower=0.0) if et_col else np.nan
    return df, et_col


def ffp_valid_rows(df: pd.DataFrame) -> int:
    """Rows passing the FFP input filter and Kljun et al. (2015) validity bounds."""
    zm, z0, ol = df[COLUMNS["zm"]], df[COLUMNS["z0"]], df[COLUMNS["ol"]]
    h = df[BL_HEIGHT] if isinstance(BL_HEIGHT, str) else BL_HEIGHT
    z0_factor = 20.0 if FFP_RSLAYER else 27.5
    ok = (df[COLUMNS["ustar"]] > 0.1) & (zm / ol >= -15.5) & (zm > z0_factor * z0) & (zm < 0.8 * h)
    return int(ok.sum())


def regrid(fclim: xr.DataArray) -> np.ndarray:
    """Climatology on the common grid, dims (y, x), integrating to 1."""
    out = fclim.transpose("x", "y").interp(
        x=common_x, y=common_y, method="linear", kwargs={"fill_value": 0.0}
    ).fillna(0.0)
    arr = np.clip(out.values, 0.0, None)   # interpolation can create tiny negative ripples
    total = arr.sum() * cell_area
    if total > 0:
        arr = arr / total
    return arr.T


def write_combined_store(sid, lat, lon, climatologies, store) -> Path:
    """climatologies: {model_type: {month_start (Timestamp): (y, x) array}}"""
    months = sorted({m for per_model in climatologies.values() for m in per_model})
    data_vars = {}
    for model_type, per_month in climatologies.items():
        stack = np.full((len(months), len(common_y), len(common_x)), np.nan, dtype="float32")
        for i, month in enumerate(months):
            if month in per_month:
                stack[i] = per_month[month]
        data_vars[model_type] = xr.DataArray(
            stack,
            dims=("time_monthly", "y", "x"),
            attrs={
                "long_name": f"Monthly {model_type} footprint climatology (regridded)",
                "units": "m-2",
                "grid_mapping": "spatial_ref",
            },
        )

    utm_epsg = choose_utm_epsg(lon, lat)
    utm_crs = CRS.from_epsg(utm_epsg)
    x0, y0 = Transformer.from_crs(4326, utm_crs, always_xy=True).transform(lon, lat)

    ds = xr.Dataset(
        data_vars,
        coords={
            "time_monthly": pd.DatetimeIndex(months),
            "x": ("x", common_x, {"units": "m", "long_name": "Distance east of tower"}),
            "y": ("y", common_y, {"units": "m", "long_name": "Distance north of tower"}),
            "easting": ("x", common_x + x0, {"units": "m", "standard_name": "projection_x_coordinate"}),
            "northing": ("y", common_y + y0, {"units": "m", "standard_name": "projection_y_coordinate"}),
            "spatial_ref": xr.DataArray(
                np.int32(0),
                attrs={**utm_crs.to_cf(), "crs_wkt": utm_crs.to_wkt(), "epsg_code": f"EPSG:{utm_epsg}"},
            ),
        },
        attrs={
            "title": f"{sid} monthly footprint climatology by model",
            "station": sid,
            "station_lat": float(lat),
            "station_lon": float(lon),
            "tower_easting": float(x0),
            "tower_northing": float(y0),
            "crs": f"EPSG:{utm_epsg}",
            "models": list(climatologies),
            "dx": DX,
            "dy": DY,
            "daytime_hours": f"{START_TIME}-{END_TIME}",
            "Conventions": "CF-1.8",
            "source": "fluxfootprints",
        },
    )
    write_zarr(ds, store)
    return Path(store)

## Run Every Model for Every Station

Loop order is station -> month -> model. Every run is wrapped in `try`/`except`, so one model
failing for one month (e.g. `wang` with no unstable rows) is recorded in the manifest and the
loop moves on. A run whose climatology comes back empty or containing NaN also counts as
failed, and nothing is written for it. Existing stores are overwritten.

In [ ]:
manifest = []         # one row per station/month/model
combined_stores = {}  # station -> path of the combined monthly climatology store
t_start = time.perf_counter()

for row in stations_df.itertuples():
    sid = row.station
    df_station, et_col = load_station(row)
    print(f"\n=== {sid} ({row.lat:.4f}, {row.lon:.4f}), ET weights from {et_col} ===")
    if df_station.empty:
        print(f"  no data between {START} and {END}, skipping")
        continue

    station_dir = OUT_DIR / sid
    climatologies = {m: {} for m in MODEL_TYPES}   # model -> {month_start: (y, x) array}

    for period in pd.period_range(df_station.index.min(), df_station.index.max(), freq="M"):
        label = str(period)
        yyyymm = label.replace("-", "")
        df_mon = df_station.loc[label].dropna(subset=REQ_FIELDS)
        n_unstable = int((df_mon[COLUMNS["ol"]] < 0).sum())
        n_ffp = ffp_valid_rows(df_mon)
        print(f"  {label}: {len(df_mon)} valid rows ({n_unstable} unstable, {n_ffp} within FFP bounds)")

        for model_type in MODEL_TYPES:
            rec = {"station": sid, "month": label, "model": model_type,
                   "n_input_rows": len(df_mon), "n_ffp_valid": n_ffp,
                   "et_source": et_col, "status": "ok",
                   "results_store": None, "summaries_store": None}
            model_dir = station_dir / model_type
            model_dir.mkdir(parents=True, exist_ok=True)
            model = summaries = None
            t0 = time.perf_counter()
            try:
                if len(df_mon) < MIN_VALID_TIMESTEPS:
                    raise ValueError(f"fewer than {MIN_VALID_TIMESTEPS} valid rows")

                df_in = df_mon
                if model_type == "ls" and LS_MAX_TIMESTEPS and len(df_mon) > LS_MAX_TIMESTEPS:
                    keep = np.linspace(0, len(df_mon) - 1, LS_MAX_TIMESTEPS).round().astype(int)
                    df_in = df_mon.iloc[keep]

                model = build_climatology(
                    df_in,
                    model_type=model_type,
                    **COLUMNS,
                    h=BL_HEIGHT,
                    dx=DX,
                    dy=DY,
                    domain=DOMAIN,
                    verbosity=0,
                    **({"n_particles": LS_N_PARTICLES} if model_type == "ls" else {}),
                    **({"rslayer": FFP_RSLAYER} if model_type in SUMMARY_MODELS else {}),
                )
                rec["n_timesteps"] = getattr(model, "ts_len", None) or model.results.attrs.get("n_timesteps")
                fclim = model.get_footprint_climatology().values
                if not np.isfinite(fclim).all() or fclim.sum() <= 0:
                    detail = (f" ({ffp_valid_rows(df_in)} of {len(df_in)} rows within FFP bounds)"
                              if model_type in SUMMARY_MODELS else "")
                    raise ValueError(f"model returned an empty or NaN climatology{detail}")

                if WRITE_FULL_RESULTS:
                    store = model_dir / f"{sid}_{model_type}_{yyyymm}_results.zarr"
                    model.to_zarr(store)
                    rec["results_store"] = str(store)

                if WRITE_SUMMARIES and model_type in SUMMARY_MODELS:
                    has_et = df_mon["et_weight"].notna().any()
                    summaries = summarize_periods(
                        model,
                        df_mon,
                        et_source="et_weight" if has_et else None,
                        is_le=False,
                        calc_et_weighted=has_et,
                        daily=True,
                        monthly=True,
                        normalize_each_time=True,
                    )
                    store = model_dir / f"{sid}_{model_type}_{yyyymm}_summaries.zarr"
                    export_summaries_zarr(
                        model, summaries,
                        station_lat=row.lat, station_lon=row.lon,
                        store=store,
                    )
                    rec["summaries_store"] = str(store)
                    cov = summaries.monthly_domain_coverage
                    rec["mean_coverage_pct"] = float(cov["mean_coverage_pct"].iloc[0])

                climatologies[model_type][period.to_timestamp()] = regrid(model.get_footprint_climatology())

            except Exception as exc:
                rec["status"] = f"failed: {exc}"
            finally:
                rec["runtime_s"] = round(time.perf_counter() - t0, 2)
                manifest.append(rec)
                print(f"    {model_type:7s} {rec['status'][:70]:70s} {rec['runtime_s']:7.1f}s")
                del model, summaries
                gc.collect()

    climatologies = {m: c for m, c in climatologies.items() if c}
    if WRITE_COMBINED and climatologies:
        store = station_dir / f"{sid}_monthly_climatology.zarr"
        combined_stores[sid] = write_combined_store(sid, row.lat, row.lon, climatologies, store)
        print(f"  combined store: {store}")

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUT_DIR / "run_manifest.csv", index=False)
print(f"\nFinished in {time.perf_counter() - t_start:.0f}s; manifest at {OUT_DIR / 'run_manifest.csv'}")

## Review the Run

In [ ]:
ok = manifest_df["status"] == "ok"
print(f"{ok.sum()} of {len(manifest_df)} station/month/model runs succeeded")
manifest_df.pivot_table(index=["station", "month"], columns="model", values="status",
                        aggfunc=lambda s: "ok" if (s == "ok").all() else "failed")

In [ ]:
# Failure reasons, if any
manifest_df.loc[~ok, ["station", "month", "model", "n_input_rows", "status"]]

In [ ]:
# Runtime per model, summed over stations and months
manifest_df.groupby("model")["runtime_s"].sum().reindex(MODEL_TYPES).round(1)

## Check the Zarr Output

Open one station's combined store lazily with `xarray.open_zarr` and plot every model's
climatology for the first month. Each panel has its own (square-root) colour scale, and the view
is zoomed to 1.5x the area holding 80% of the footprint. Because all models share one grid, the variables can be
compared or differenced directly. The `easting`/`northing` coordinates and the `spatial_ref`
variable place the grid in UTM (e.g. `ds.rio.write_crs(ds.attrs["crs"])` with rioxarray).

In [ ]:
sid = next(iter(combined_stores))
ds = xr.open_zarr(combined_stores[sid])
ds

In [ ]:
month = ds["time_monthly"].values[0]
layers = {m: ds[m].sel(time_monthly=month).values for m in ds.data_vars}
layers = {m: a for m, a in layers.items() if np.isfinite(a).any()}   # models with data this month


def extent(arr, frac=0.8):
    # largest |x| or |y| of the cells holding `frac` of the footprint
    flat = np.sort(np.nan_to_num(arr).ravel())[::-1]
    n = min(np.searchsorted(np.cumsum(flat) / flat.sum(), frac), flat.size - 1)
    yy, xx = np.nonzero(arr >= flat[n])
    return max(np.abs(ds["x"].values[xx]).max(), np.abs(ds["y"].values[yy]).max())


lim = 1.5 * max(extent(a) for a in layers.values()) + DX

fig, axes = plt.subplots(1, len(layers), figsize=(4.4 * len(layers), 4.0), squeeze=False, layout="constrained")
axes = axes[0]
for ax, (m, arr) in zip(axes, layers.items()):
    im = ax.pcolormesh(ds["x"], ds["y"], arr, cmap="viridis", norm=PowerNorm(0.5, vmin=0), shading="auto")
    ax.plot(0, 0, marker="^", color="white", markeredgecolor="black", markersize=9, zorder=5)
    ax.set_title(m)
    ax.set_xlabel("x [m]")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    fig.colorbar(im, ax=ax, shrink=0.8)
axes[0].set_ylabel("y [m]")
fig.suptitle(f"{sid} footprint climatology, {pd.Timestamp(month):%Y-%m} (density, m$^{{-2}}$)")
plt.show()

In [ ]:
# A per-month summaries store (ffp/ffp_xr): daily and monthly layers on separate time axes
summary_stores = manifest_df["summaries_store"].dropna()
xr.open_zarr(summary_stores.iloc[0]) if len(summary_stores) else "No summaries stores written"

## Notes & Caveats

- **Re-running overwrites** every store for the stations and months processed. Point `OUT_DIR`
  somewhere new to keep an earlier run.
- **Combined store vs. per-month stores.** The combined store holds regridded, renormalized
  climatologies, which suits model and month comparisons. Use the per-month `results`/
  `summaries` stores when you need each model's native grid or the daily layers.
- **Summaries use `normalize_each_time=True`**, as in
  [footprint_workflow_example.ipynb](footprint_workflow_example.ipynb), so daily/monthly
  values there are per-cell weights summing to 1, not densities.
- **ET weighting** uses the first column in `ET_WEIGHT_COLUMNS` with data in the analysis
  period (`nldas_et`, then tower `et_1_1_1`), clipped at 0; the column used is in the
  manifest's `et_source`. Without either, the ET-weighted layers are left out of the
  summaries stores.
- **Empty `ffp`/`ffp_xr` climatologies** mean no row passed the Kljun et al. (2015) bounds.
  Compare `n_ffp_valid` with `n_input_rows` in the manifest. The usual cause is the
  roughness-sublayer check failing for a tall crop or low sensor, so check
  `measurement_height`, `crop_height_m` and `VEG_TYPE`.
- **Tall crops.** With the alfalfa preset (d = 0.67 h_c, z0 = 0.123 h_c) and sensor height H,
  FFP rows are valid only while h_c < H / 4.05 with `FFP_RSLAYER = False`, or h_c < H / 3.13
  with `FFP_RSLAYER = True` (0.60 m and 0.78 m for H = 2.44 m). Above that, FFP is outside its
  validity range (zm > 20 z0, Kljun et al. 2015 Eq. 27), and those rows are dropped. `km`,
  `ls` and `wang` do not apply this check, so they still give a footprint for those periods.
- **`ls` is stochastic and subsampled**, so its stores change slightly between runs and its
  monthly climatology rests on at most `LS_MAX_TIMESTEPS` rows (`n_timesteps` in the manifest).
- **Store size.** `WRITE_FULL_RESULTS` keeps the per-timestep `(time, x, y)` stack for
  `ffp`/`ffp_xr`, which dominates disk use over long periods. Set it to `False` if the
  summaries and combined stores are enough.